# FIAP – Fase 6 | Capítulo 1 – FarmTech Solutions
## Visão Computacional com YOLOv5 (Detecção de Objetos)
**Autor:** Deivisson Gonçalves Lima – **RM565095**  
**Grupo:** 47 (Trabalho individual)  
**Notebook:** `DeivissonLima_RM565095_fase6_cap1_colab_v3.ipynb`

**Classes:** `copo` (Coffee cup) e `controle` (Remote control)


## 1) Montar Drive e paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/Fase6/Fase6_Cap1'
DATA_DIR_DRIVE = f"{DRIVE_BASE}/data"  # backup opcional
DATA_DIR_LOCAL = '/content/fase6_data'  # dataset local
RUNS_DIR = f"{DRIVE_BASE}/runs"
print('DRIVE_BASE =', DRIVE_BASE)
print('DATA_DIR_LOCAL =', DATA_DIR_LOCAL)
print('RUNS_DIR =', RUNS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_BASE = /content/drive/MyDrive/Fase6/Fase6_Cap1
DATA_DIR_LOCAL = /content/fase6_data
RUNS_DIR = /content/drive/MyDrive/Fase6/Fase6_Cap1/runs


## 2) Clonar YOLOv5 e dependências

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip -q install -r requirements.txt
import torch
print('CUDA disponível?', torch.cuda.is_available())
print('Torch:', torch.__version__)


fatal: destination path 'yolov5' already exists and is not an empty directory.
/content/yolov5
CUDA disponível? False
Torch: 2.8.0+cu126


## 3) Instalar FiftyOne com versões compatíveis (reinicie o runtime se for solicitado)

In [ ]:
%pip -q uninstall -y fiftyone fiftyone-db fiftyone-brain fiftyone-plugins mongoengine pymongo motor sse-starlette starlette
%pip -q install 'fiftyone==0.25.0' 'mongoengine==0.24.2' 'pymongo==4.6.3' 'motor==3.4.0' pyyaml
import fiftyone as fo, mongoengine, pymongo
print('fiftyone:', fo.__version__, '| mongoengine:', mongoengine.__version__, '| pymongo:', pymongo.version)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.16.0 requires sse-starlette>=1.6.1, but you have sse-starlette 0.10.3 which is incompatible.
fiftyone: 0.25.0 | mongoengine: 0.24.2 | pymongo: 4.6.3


## 4) Baixar dataset (Open Images V7 via FiftyOne) e detectar campo de rótulos

In [ ]:
import os, random, yaml, fiftyone as fo, fiftyone.zoo as foz
from fiftyone import ViewField as F

CLASSES = ['Coffee cup','Remote control']
ds = foz.load_zoo_dataset(
    'open-images-v7', split='train', label_types=['detections'],
    classes=CLASSES, only_matching=True, max_samples=3000, shuffle=True, seed=51,
)
schema = ds.get_field_schema()
det_field = None
for fname, ftype in schema.items():
    if 'Detections' in str(ftype):
        det_field = fname; break
assert det_field, f'Nenhum campo Detections no schema: {schema}'
print('det_field:', det_field)
base = ds.filter_labels(det_field, F('label').is_in(CLASSES))
for cls in CLASSES:
    v = base.filter_labels(det_field, F('label') == cls).match(F(f'{det_field}.detections').length() > 0)
    print(cls, 'amostras:', len(v))


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


Necessary images already downloaded


INFO:fiftyone.utils.openimages:Necessary images already downloaded


Existing download of split 'train' is sufficient


INFO:fiftyone.zoo.datasets:Existing download of split 'train' is sufficient


...

 100% |███████████████| 3000/3000 [7.9s elapsed, 0s remaining, 277.8 samples/s]       


INFO:eta.core.utils: 100% |███████████████| 3000/3000 [7.9s elapsed, 0s remaining, 277.8 samples/s]       


Dataset 'open-images-v7-train-3000' created


INFO:fiftyone.zoo.datasets:Dataset 'open-images-v7-train-3000' created


det_field: ground_truth
Coffee cup amostras: 2862
Remote control amostras: 138


## 5) Selecionar 40/40 por classe e montar splits 32/4/4

In [ ]:
def pick_ids_for_class(view, class_name, k, exclude=set(), seed=7):
    v = (view.filter_labels(det_field, F('label') == class_name)
            .match(F(f'{det_field}.detections').length() > 0)
            .shuffle(seed=seed))
    ids=[]
    for _id in v.values('id'):
        if _id not in exclude:
            ids.append(_id)
        if len(ids) >= k: break
    return ids

ids_A = pick_ids_for_class(base, 'Coffee cup', 40, set(), 11)
ids_B = pick_ids_for_class(base, 'Remote control', 40, set(ids_A), 13)
print('ids_A:', len(ids_A), '| ids_B:', len(ids_B))
assert len(ids_A)==40 and len(ids_B)==40, 'Aumente max_samples no load_zoo_dataset'

def split_32_4_4(ids):
    r = ids[:]; import random; random.Random(99).shuffle(r)
    return r[:32], r[32:36], r[36:40]

train_A,val_A,test_A = split_32_4_4(ids_A)
train_B,val_B,test_B = split_32_4_4(ids_B)
train_ids = train_A + train_B
val_ids   = val_A   + val_B
test_ids  = test_A  + test_B

train_view = base.select(train_ids).filter_labels(det_field, F('label').is_in(CLASSES))
val_view   = base.select(val_ids).filter_labels(det_field, F('label').is_in(CLASSES))
test_view  = base.select(test_ids).filter_labels(det_field, F('label').is_in(CLASSES))
print('Views prontas.')


ids_A: 40 | ids_B: 40
Views prontas.


## 6) Exportar para YOLOv5 (local, 1 pasta por split)

In [ ]:
import shutil, glob, yaml, fiftyone as fo
DATA_DIR_LOCAL = '/content/fase6_data'
shutil.rmtree(DATA_DIR_LOCAL, ignore_errors=True)
os.makedirs(DATA_DIR_LOCAL, exist_ok=True)

def export_split(view, split):
    outdir = os.path.join(DATA_DIR_LOCAL, split)
    os.makedirs(os.path.join(outdir,'images'), exist_ok=True)
    os.makedirs(os.path.join(outdir,'labels'), exist_ok=True)
    view.export(
        export_dir=outdir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=det_field,
        classes=CLASSES,
        export_media=True,
        overwrite=True,
    )

export_split(train_view, 'train')
export_split(val_view,   'val')
export_split(test_view,  'test')

for s in ['train','val','test']:
    imgs = glob.glob(f"{DATA_DIR_LOCAL}/{s}/images/*")
    lbls = glob.glob(f"{DATA_DIR_LOCAL}/{s}/labels/*")
    print(s, 'imgs:', len(imgs), 'labels:', len(lbls))


 100% |███████████████████| 64/64 [514.7ms elapsed, 0s remaining, 124.3 samples/s]      


INFO:eta.core.utils: 100% |███████████████████| 64/64 [514.7ms elapsed, 0s remaining, 124.3 samples/s]      


 100% |█████████████████████| 8/8 [75.7ms elapsed, 0s remaining, 105.7 samples/s] 


INFO:eta.core.utils: 100% |█████████████████████| 8/8 [75.7ms elapsed, 0s remaining, 105.7 samples/s] 


 100% |█████████████████████| 8/8 [92.8ms elapsed, 0s remaining, 86.2 samples/s] 


INFO:eta.core.utils: 100% |█████████████████████| 8/8 [92.8ms elapsed, 0s remaining, 86.2 samples/s] 


train imgs: 1 labels: 1
val imgs: 1 labels: 1
test imgs: 1 labels: 1


## 7) Gerar `data.yaml` local

In [ ]:
import yaml
DATA_YAML_LOCAL = '/content/fase6_data/data.yaml'
with open(DATA_YAML_LOCAL, 'w') as f:
    yaml.safe_dump({
        'train': '/content/fase6_data/train/images',
        'val':   '/content/fase6_data/val/images',
        'test':  '/content/fase6_data/test/images',
        'nc': 2,
        'names': ['copo','controle'],
    }, f, sort_keys=False)
!sed -n '1,120p' /content/fase6_data/data.yaml


train: /content/fase6_data/train/images
val: /content/fase6_data/val/images
test: /content/fase6_data/test/images
nc: 2
names:
- copo
- controle


## 8) Treino – 30 épocas

In [ ]:
# %cd /content/yolov5
# !python train.py --img 640 --batch 16 --epochs 30 --data /content/fase6_data/data.yaml --weights yolov5s.pt --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e30 --exist-ok


/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-10-16 00:13:49.914403: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760573630.203636   15090 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760573630.277446   15090 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760573630.691433   15090 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760573630.691486   15090 computation_placer.cc:177] computation placer a

In [ ]:
%cd /content/yolov5
!python train.py \
  --img 448 --batch 8 --epochs 30 \
  --data /content/fase6_data/data.yaml \
  --weights yolov5n.pt \
  --cache ram --workers 2 --device cpu \
  --freeze 10 --patience 100 \
  --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs \
  --name exp_e30_fast_cpu --exist-ok


/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-10-16 01:08:25.698739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760576906.085351   28464 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760576906.186774   28464 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760576906.905828   28464 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760576906.905915   28464 computation_placer.cc:177] computation placer a

## 9) Treino – 60 épocas

In [ ]:
# %cd /content/yolov5
# !python train.py --img 640 --batch 16 --epochs 60 --data /content/fase6_data/data.yaml --weights yolov5s.pt --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e60 --exist-ok


/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-10-16 00:31:25.532578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760574685.898169   19361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760574685.977215   19361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760574686.627100   19361 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760574686.627180   19361 computation_placer.cc:177] computation placer a

In [ ]:
%cd /content/yolov5
!python train.py \
  --img 448 --batch 8 --epochs 60 \
  --data /content/fase6_data/data.yaml \
  --weights yolov5s.pt \
  --cache ram --workers 2 --device cpu \
  --freeze 10 --patience 100 \
  --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs \
  --name exp_e60_fast_cpu --exist-ok


/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-10-16 01:19:23.812746: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760577563.874136   31193 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760577563.892172   31193 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760577563.937542   31193 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760577563.937627   31193 computation_placer.cc:177] computation placer a

## 10) Validação e Inferência

In [ ]:
%cd /content/yolov5
!python val.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30_fast_cpu/weights/best.pt --data /content/fase6_data/data.yaml --task val --device cpu --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e30_val --exist-ok

!python val.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt --data /content/fase6_data/data.yaml --task val --device cpu --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e60_val --exist-ok

!python detect.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt --img 448 --conf 0.25 --source /content/fase6_data/val/images --device cpu --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name infer_val_e60 --exist-ok


/content/yolov5
val: data=/content/fase6_data/data.yaml, weights=['/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30_fast_cpu/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=cpu, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/Fase6/Fase6_Cap1/runs, name=exp_e30_val, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-441-g15c0127a Python-3.12.12 torch-2.8.0+cu126 CPU

Fusing layers... 
Model summary: 157 layers, 1761871 parameters, 0 gradients, 4.1 GFLOPs
val: Scanning /content/fase6_data/val/labels/val.cache... 8 images, 0 backgrounds, 0 corrupt: 100% 8/8 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 1/1 [00:02<00:00,  2.14s/it]
                   all          8         10      0.425      0.542      0.413     0.0948
                  copo          8

In [ ]:
%cd /content/yolov5
!python detect.py \
  --weights "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60/weights/best.pt" \
  --img 640 --conf 0.25 \
  --source /content/fase6_data/val/images \
  --project "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs" \
  --name infer_val_e60 --exist-ok


/content/yolov5
detect: weights=['/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60/weights/best.pt'], source=/content/fase6_data/val/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/Fase6/Fase6_Cap1/runs, name=infer_val_e60, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-441-g15c0127a Python-3.12.12 torch-2.8.0+cu126 CPU

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Traceback (most recent call last):
  File "/content/yolov5/detect.py", line 438, in <module>
    main(opt)
  File "/content/yolov5/detect.py", line 433, in main
    run(**vars(opt))
  File "/usr/local/lib/python3.12/dist-packages/t

In [ ]:
import glob, os

paths = {
    "val_images": glob.glob("/content/fase6_data/val/images/*"),
    "val_images_deep": glob.glob("/content/fase6_data/val/images/val/*"),
    "val_labels": glob.glob("/content/fase6_data/val/labels/*"),
}
{k: len(v) for k,v in paths.items()}


{'val_images': 1, 'val_images_deep': 8, 'val_labels': 2}

In [ ]:
import os, shutil, glob

BASE = "/content/fase6_data"
def flatten_split(split):
    root = os.path.join(BASE, split)
    for sub in ["images","labels"]:
        deep = os.path.join(root, sub, split)   # ex.: /val/images/val
        if os.path.isdir(deep):
            for fn in os.listdir(deep):
                src = os.path.join(deep, fn)
                dst = os.path.join(root, sub, fn)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(deep, ignore_errors=True)

for s in ["train","val","test"]:
    flatten_split(s)

# conferir após o flatten
print("val images:", len(glob.glob("/content/fase6_data/val/images/*")))
print("val labels:", len(glob.glob("/content/fase6_data/val/labels/*")))


val images: 8
val labels: 9


In [ ]:
import os, glob

VAL_IMG_DIR = "/content/fase6_data/val/images"
VAL_LBL_DIR = "/content/fase6_data/val/labels"

imgs = sorted([p for p in glob.glob(f"{VAL_IMG_DIR}/*") if os.path.splitext(p)[1].lower() in (".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff")])
lbls = sorted(glob.glob(f"{VAL_LBL_DIR}/*.txt"))

img_basenames = {os.path.splitext(os.path.basename(p))[0] for p in imgs}
lbl_basenames = {os.path.splitext(os.path.basename(p))[0] for p in lbls}

missing_lbl = sorted(img_basenames - lbl_basenames)
orphan_lbl  = sorted(lbl_basenames - img_basenames)

print(f"Imagens: {len(imgs)} | Labels: {len(lbls)}")
print("Sem label:", missing_lbl)
print("Labels órfãs:", orphan_lbl)

# (opcional) deletar labels órfãs para evitar warnings
for b in orphan_lbl:
    p = os.path.join(VAL_LBL_DIR, b + ".txt")
    if os.path.exists(p):
        os.remove(p)

print("Depois de limpar órfãos -> Labels:", len(glob.glob(f'{VAL_LBL_DIR}/*.txt')))


Imagens: 8 | Labels: 8
Sem label: []
Labels órfãs: []
Depois de limpar órfãos -> Labels: 8


In [ ]:
# remover imagens sem label para manter 1:1
removed = 0
for b in missing_lbl:
    for ext in (".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"):
        p = os.path.join(VAL_IMG_DIR, b+ext)
        if os.path.exists(p):
            os.remove(p); removed += 1; break
print("Imagens removidas por falta de label:", removed)

import glob
print("val images agora:", len(glob.glob(f"{VAL_IMG_DIR}/*")))
print("val labels agora:", len(glob.glob(f"{VAL_LBL_DIR}/*.txt")))


Imagens removidas por falta de label: 0
val images agora: 8
val labels agora: 8


In [ ]:
# apagar caches antigos
for p in glob.glob("/content/fase6_data/*/labels/*.cache"):
    os.remove(p)

# validar e detectar novamente
%cd /content/yolov5
!python val.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt \
  --data /content/fase6_data/data.yaml --task val --device cpu \
  --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e60_val --exist-ok

!python detect.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt \
  --img 448 --conf 0.25 --source "/content/fase6_data/val/images" --device cpu \
  --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name infer_val_e60 --exist-ok


/content/yolov5
val: data=/content/fase6_data/data.yaml, weights=['/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=cpu, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/Fase6/Fase6_Cap1/runs, name=exp_e60_val, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-441-g15c0127a Python-3.12.12 torch-2.8.0+cu126 CPU

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/fase6_data/val/labels... 8 images, 0 backgrounds, 0 corrupt: 100% 8/8 [00:00<00:00, 309.70it/s]
val: New cache created: /content/fase6_data/val/labels.cache
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 1/1 [00:06<00:00,  6.25s/it]
                   all          8         10      0.601      

In [ ]:
import shutil, os, glob
SRC = "/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/infer_val_e60"
DST = "/content/drive/MyDrive/Fase6/Fase6_Cap1/prints_val_e60"
os.makedirs(DST, exist_ok=True)
for p in glob.glob(f"{SRC}/*.*"):
    shutil.copy(p, DST)
print("Copiados para:", DST)


Copiados para: /content/drive/MyDrive/Fase6/Fase6_Cap1/prints_val_e60


In [ ]:
%cd /content/yolov5
!python detect.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt \
  --img 448 --conf 0.25 --source /content/fase6_data/test/images \
  --device cpu --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name infer_test_e60 --exist-ok


/content/yolov5
detect: weights=['/content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e60_fast_cpu/weights/best.pt'], source=/content/fase6_data/test/images, data=data/coco128.yaml, imgsz=[448, 448], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=cpu, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/Fase6/Fase6_Cap1/runs, name=infer_test_e60, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-441-g15c0127a Python-3.12.12 torch-2.8.0+cu126 CPU

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Traceback (most recent call last):
  File "/content/yolov5/detect.py", line 438, in <module>
    main(opt)
  File "/content/yolov5/detect.py", line 433, in main
    run(**vars(opt))
  File "/usr/local/lib/python3.12/d

## 11) Análise e Conclusões
- Compare mAP@0.5 / mAP@0.5:0.95 (30 vs 60)
- Avalie precision/recall e losses
- Inclua prints de `infer_test_e60`
- Limitações e próximos passos (augmentation, mais dados, tuning)
